# Movie Recommendation System using TMDb Dataset

This project develops a content-based movie recommendation system using the TMDb movie dataset. The system recommends movies similar to a selected movie using TF-IDF Vectorization and Cosine Similarity.

In [19]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [20]:
df = pd.read_csv("./tmdb_5000_movies.csv")
df.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


## Dataset Exploration
Understanding the dataset structure and checking for missing values.

In [21]:
df.info()
df.shape
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   str    
 2   homepage              1712 non-null   str    
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   str    
 5   original_language     4803 non-null   str    
 6   original_title        4803 non-null   str    
 7   overview              4800 non-null   str    
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   str    
 10  production_countries  4803 non-null   str    
 11  release_date          4802 non-null   str    
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   str    
 15  status                4803 non-n

budget                     0
genres                     0
homepage                3091
id                         0
keywords                   0
original_language          0
original_title             0
overview                   3
popularity                 0
production_companies       0
production_countries       0
release_date               1
revenue                    0
runtime                    2
spoken_languages           0
status                     0
tagline                  844
title                      0
vote_average               0
vote_count                 0
dtype: int64

## Feature Selection

The recommendation system will use movie titles and overviews.

In [22]:
movies = df[
    [
        "title",
        "overview"
    ]
]
movies.head()

,title,overview
0,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."
2,Spectre,A cryptic message from Bond’s past sends him o...
3,The Dark Knight Rises,Following the death of District Attorney Harve...
4,John Carter,"John Carter is a war-weary, former military ca..."


## Data Cleaning

Missing overview values are replaced with empty strings.

In [23]:
movies["overview"] = movies[
    "overview"
].fillna("")

## TF-IDF Vectorization

Movie overviews are converted into numerical vectors.

In [24]:
tfidf = TfidfVectorizer(
    stop_words="english"
)

tfidf_matrix = tfidf.fit_transform(
    movies["overview"]
)
print(tfidf_matrix.shape)

(4803, 20978)


## Similarity Calculation

Cosine similarity is used to measure how similar two movies are.

In [25]:
cosine_sim = cosine_similarity(
    tfidf_matrix,
    tfidf_matrix
)
print(cosine_sim.shape)

(4803, 4803)


In [26]:
indices = pd.Series(
    movies.index,
    index=movies["title"]
)
indices = indices.drop_duplicates()

## Recommendation Function

This function recommends the top 5 movies similar to a given movie.

In [27]:
def recommend_movies(title):

    idx = indices[title]

    sim_scores = list(
        enumerate(cosine_sim[idx])
    )

    sim_scores = sorted(
        sim_scores,
        key=lambda x: x[1],
        reverse=True
    )

    sim_scores = sim_scores[1:6]

    movie_indices = [
        i[0]
        for i in sim_scores
    ]

    recommended_movies = movies["title"].iloc[movie_indices]

    print("Recommended Movies:\n")

    for movie in recommended_movies:
        print(movie)

## Testing Recommendations

In [28]:
recommend_movies(
    "Avatar"
)
recommend_movies(
    "The Dark Knight"
)
recommend_movies(
    "Titanic"
)

Recommended Movies:

Apollo 18
The American
The Matrix
The Inhabited Island
Tears of the Sun
Recommended Movies:

The Dark Knight Rises
Batman Returns
Batman: The Dark Knight Returns, Part 2
Batman Forever
Batman
Recommended Movies:

Raise the Titanic
Ghost Ship
I Can Do Bad All By Myself
Event Horizon
Niagara


In [29]:
print(
    recommend_movies(
        "Avatar"
    ),
   recommend_movies(
    "The Dark Knight"
),
   recommend_movies(
    "Titanic"
)
)

Recommended Movies:

Apollo 18
The American
The Matrix
The Inhabited Island
Tears of the Sun
Recommended Movies:

The Dark Knight Rises
Batman Returns
Batman: The Dark Knight Returns, Part 2
Batman Forever
Batman
Recommended Movies:

Raise the Titanic
Ghost Ship
I Can Do Bad All By Myself
Event Horizon
Niagara
None None None


## Conclusion

A content-based movie recommendation system was developed using the TMDb dataset.

Movie descriptions were converted into numerical vectors using TF-IDF Vectorization. Cosine Similarity was used to measure similarity between movies and generate recommendations.

The system successfully recommends movies based on content similarity and demonstrates the practical application of recommendation algorithms in data science.